# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [1]:
import os
import time
from rtm_pymmcore.data_structures import Fov, Channel, StimTreatment
from pprint import pprint
import pandas as pd
import numpy as np
import dataclasses
import random

### Experimental Settings

In [2]:
# from rtm_pymmcore.microscope.MMDemo import MMDemo

# mic = MMDemo()
# mic.mmc.setChannelGroup("Channel")

# for Jungfrau Microscope enable here:
# from rtm_pymmcore.microscope.Jungfrau import Jungfrau

# mic = Jungfrau()
# mic.mmc.setChannelGroup("TTL_ERK")

from rtm_pymmcore.microscope.Niesen import Niesen
mic = Niesen()
mic.mmc.setChannelGroup("WF_TTL")


In [3]:
## Configuration options
FIRST_FRAME_STIMULATION = 1
# N_FRAMES = 340
N_FRAMES_PHASE_1 = 2
N_FRAMES_PHASE_2 = 3


SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

TIME_BETWEEN_TIMESTEPS = 5  # time in seconds between frames
TIME_PER_FOV = 3.3  # time in seconds per fov

ADD_STIM_EXPOSURE_GROUP = (
    False  # add an entry showing the last stimulation exposure for each FOV
)
REGULAR_SPACING_BETWEEN_STIMULATIONS = (
    False  # if True, the stim_timesteps will be evenly spaced
)


## Storage path for the experiment
base_path = "E:\\Alex"
experiment_name = "Test_1"
path = os.path.join(base_path, experiment_name)

path_with_old_data_for_simulation = os.path.join(
    ".", "test_exp_data", "00_01_demo_imgs"
)  # path to the folder with old data for simulation


# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
channels = []
channels.append(Channel(name="Red", exposure=100))

# Channel to check for the expression of the optogenetic marker, can be used if it the marker is in the same channel as the stimulation channel.
channel_optocheck = Channel(name="Red", exposure=300)
optocheck_timepoints = (3,)  # timepoints in frames when optocheck is done


condition = ["test"]

# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24 # Example of adding multiple conditions to the dataframe. n repreats the amount of times the condition is repeated.


n_fovs_per_well = None  ## change this variable to the amount of fovs that you have per well. Set to None if you are not working with wellplate.


# Stimulation parameters for optogenetics. The stimulation will be repeated for each condition.

stim_exposures_timesteps_phase_1 = [

]

stim_exposures_timesteps_phase_2 = [

]


def fix_tuples_in_stim_exposure_list(
    stim_exposures_timesteps,
):
    """Convert any range or list in the stim_exposures_timesteps_before_pause to tuples."""

    for stim_exposure_timestep in stim_exposures_timesteps:
        if isinstance(stim_exposure_timestep["stim_timestep"], range):
            stim_exposure_timestep["stim_timestep"] = tuple(
                stim_exposure_timestep["stim_timestep"]
            )
        if isinstance(stim_exposure_timestep["stim_exposure_list"], range):
            stim_exposure_timestep["stim_exposure_list"] = tuple(
                stim_exposure_timestep["stim_exposure_list"]
            )
        if isinstance(stim_exposure_timestep["stim_timestep"], list):
            stim_exposure_timestep["stim_timestep"] = tuple(
                stim_exposure_timestep["stim_timestep"]
            )
        if isinstance(stim_exposure_timestep["stim_exposure_list"], list):
            stim_exposure_timestep["stim_exposure_list"] = tuple(
                stim_exposure_timestep["stim_exposure_list"]
            )


def add_stim_parameters_to_stim_exposures_timesteps(
    stim_exposures_timesteps,
):
    """Add general stimulation parameters to each stim_exposures_timesteps_before_pause dict."""
    for stim_exposure_timestep in stim_exposures_timesteps:
        stim_exposure_timestep["stim_power"] = 10
        stim_exposure_timestep["stim_channel_name"] = "Red"
        stim_exposure_timestep["stim_channel_group"] = "WF_TTL"
        stim_exposure_timestep["stim_channel_device_name"] = "LedDMD"
        stim_exposure_timestep["stim_channel_power_property_name"] = "Cyan_Level"


def print_stim_exposures_timesteps(
    stim_exposures_timesteps,
):
    """Print the stim_exposures_timesteps_before_pause in a readable format."""
    for stim_exposure_timestep in stim_exposures_timesteps:
        print("Pattern Name: ", stim_exposure_timestep["ramp_pattern_name"])

        for stim_exp, stim_timestep in zip(
            stim_exposure_timestep["stim_exposure_list"],
            stim_exposure_timestep["stim_timestep"],
        ):
            print(f"{stim_exp} at {stim_timestep}")
        print("")


for stim_exposures_timesteps in [
    stim_exposures_timesteps_phase_1,
    stim_exposures_timesteps_phase_2,
]:
    fix_tuples_in_stim_exposure_list(stim_exposures_timesteps)
    add_stim_parameters_to_stim_exposures_timesteps(stim_exposures_timesteps)
    print_stim_exposures_timesteps(stim_exposures_timesteps)

## Define the Tools that you are using for the experiment
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.simple_fe import SimpleFE
from rtm_pymmcore.feature_extraction.optocheck_fe import OptoCheckFE
from rtm_pymmcore.segmentation.cellpose_v4 import CellposeV4

# segmentators = [
#     {
#         "name": "labels",
#         "class": CellposeV4(
#             custom_model_path="E:\\models\\cellpose\\LifeActH2B_mixed_with_only_H2B_v1",
#             min_size=100,
#         ),
#         "use_channel": 0,
#         "save_tracked": True,
#     },
# ]

segmentators = [
    {
        "name": "labels",
        "class": CellposeV4(
        ),
        "use_channel": 0,
        "save_tracked": True,
    },
]

stimulator = StimWholeFOV()
# feature_extractor = FE_ErkKtr("labels")
feature_extractor = SimpleFE("labels")
tracker = TrackerTrackpy(search_range=15)
optocheck = None


from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

Directory E:\Alex\Test_1\raw already exists
Directory E:\Alex\Test_1\tracks already exists
Directory E:\Alex\Test_1\stim_mask already exists
Directory E:\Alex\Test_1\stim already exists
Directory E:\Alex\Test_1\particles already exists
Directory E:\Alex\Test_1\labels already exists


### GUI - Napari Micromanager

#### Load GUI

In [4]:
mic.mmc.setProperty("Laser", "RED_Intensity", 100)
mic.mmc.setProperty("Camera-1", "Binning", "2x2")

In [6]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

Please execute the following code block, if you would like to set a custom ROI (e.g. smaller illumination than the full field of view of the camera). Execute it after you have started the camera once in the GUI. 

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

### Map Experiment to FOVs

### Use FOVs to generate dataframe for acquisition

In [5]:
def _get_mda_from_file(filename):
    import json

    file = os.path.join(filename)
    with open(file, "r") as f:
        data_mda_fovs = json.load(f)
    return data_mda_fovs


def _get_mda_from_viewer(viewer):
    data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions
    data_mda_fovs_dict = []
    for data_mda in data_mda_fovs:
        data_mda_fovs_dict.append(data_mda.model_dump())
    data_mda_fovs = data_mda_fovs_dict
    return data_mda_fovs


def generate_fov_objects(viewer=None, filename=None):
    if filename is not None:
        data_mda_fovs = _get_mda_from_file(filename)
    elif viewer is not None:
        data_mda_fovs = _get_mda_from_viewer(viewer)
        if data_mda_fovs is None:
            assert False, "No fovs selected. Please select fovs in the MDA widget"
    else:
        assert False, "Either viewer must be provided or from_file must be True"

    fovs = []
    for i, fov in enumerate(data_mda_fovs):
        fov_object = Fov(i)
        fov_object.x = fov.get("x")
        fov_object.y = fov.get("y")
        fov_object.z = fov.get("z") if not mic.USE_ONLY_PFS else None
        fov_object.name = str(i) if fov["name"] is None else fov["name"]
        fovs.append(fov_object)
    return fovs


def generate_df_acquire(
    fovs,
    n_frames,
    time_between_timesteps,
    time_per_fov,
    channels,
    start_time=0,
    channel_optocheck=None,
    optocheck_timepoints=None,
    phase_id=None,
    phase_name=None,
    condition=None,
):
    n_fovs_simultaneously = time_between_timesteps // time_per_fov
    optocheck_timepoints = (
        optocheck_timepoints if optocheck_timepoints is not None else [n_frames - 1]
    )
    timesteps = range(n_frames)
    dfs = []
    for fov_index, fov in enumerate(fovs):
        fov_group = fov_index // n_fovs_simultaneously
        start_time_fov = fov_group * time_between_timesteps * len(timesteps)
        if condition is None or len(condition) == 0:
            condition_fov = None
        elif len(condition) == 1:
            condition_fov = condition[0]
        else:
            condition_fov = condition[fov_index]
        for timestep in timesteps:
            if phase_id is not None:
                fname = f"{str(fov_index).zfill(3)}_{str(phase_id).zfill(2)}_{str(timestep).zfill(5)}"
            else:
                fname = f"{str(fov_index).zfill(3)}_{str(timestep).zfill(5)}"
            row = {
                "fov_object": fov,
                "fov": fov_index,
                "fov_x": fov.x,
                "fov_y": fov.y,
                "fov_z": fov.z,
                "fov_name": fov.name,
                "timestep": timestep,
                "time": start_time_fov + timestep * time_between_timesteps,
                "channels": tuple(dataclasses.asdict(channel) for channel in channels),
                "fname": fname,
            }
            if condition_fov is not None:
                row["cell_line"] = condition_fov
            if channel_optocheck is not None:
                row["optocheck"] = True if timestep in optocheck_timepoints else False
                if isinstance(channel_optocheck, list):
                    row["optocheck_channels"] = tuple(
                        dataclasses.asdict(channel) for channel in channel_optocheck
                    )
                else:
                    row["optocheck_channels"] = tuple(
                        [dataclasses.asdict(channel_optocheck)]
                    )
            dfs.append(row)

    df_acquire = pd.DataFrame(dfs)
    if phase_name is not None:
        df_acquire["phase"] = phase_name
    if phase_id is not None:
        df_acquire["phase_id"] = phase_id
    if phase_name or phase_id is not None:
        df_acquire["phase_timestep"] = df_acquire["timestep"] + int(phase_id) * n_frames
    print(f"Total Experiment Time: {df_acquire['time'].max()/3600}h")
    return df_acquire


def apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_exposures_timesteps,
    condition,
    n_fovs_per_well=None,
    add_stim_exposure_group=False,
    regular_spacing_between_stimulations=False,
):
    """Apply stim treatments to the df_acquire dataframe."""

    n_fovs = len(df_acquire["fov"].unique())
    n_stim_treatments = len(stim_exposures_timesteps)
    if n_stim_treatments > 0:
        n_fovs_per_stim_condition = (
            n_fovs // n_stim_treatments // len(np.unique(condition))
        )
        stim_treatment_tot = []
        random.shuffle(stim_exposures_timesteps)
        if n_fovs_per_well is None:
            for fov_index in range(0, n_fovs_per_stim_condition):
                stim_treatment_tot.extend(stim_exposures_timesteps)
            random.shuffle(stim_treatment_tot)

            if n_fovs % n_stim_treatments != 0:
                print(
                    f"Warning: Not equal number of fovs per stim condition. {n_fovs % n_stim_treatments} fovs will have repeated treatment"
                )
                stim_treatment_tot.extend(
                    stim_exposures_timesteps[: n_fovs % n_stim_treatments]
                )
            print(f"Doing {n_fovs_per_stim_condition} experiment per stim condition")

            if len(condition) != 1:
                stim_treatment_tot = stim_treatment_tot * len(np.unique(condition))

            df_acquire = pd.merge(
                df_acquire,
                pd.DataFrame(stim_treatment_tot),
                left_on="fov",
                right_index=True,
            )
        else:
            stim_treatment_tot = []
            for cell_line in np.unique(condition):
                fovs_for_one_cell_line = df_acquire.query(f"cell_line == @cell_line")[
                    "fov"
                ].unique()
                stim_treat = [
                    stim
                    for stim in stim_exposures_timesteps
                    for _ in range(n_fovs_per_well)
                ]
                if len(fovs_for_one_cell_line) != len(stim_treat):
                    print(
                        f"Warning: Number of fovs ({len(fovs_for_one_cell_line)}) for cell line {cell_line} does not match number of stim treatments ({len(stim_treat)})."
                    )
                stim_treat = pd.DataFrame(stim_treat)
                stim_treat["fov"] = fovs_for_one_cell_line
                stim_treatment_tot.append(stim_treat)
            stim_treat = pd.concat(stim_treatment_tot, ignore_index=True)
            df_acquire = pd.merge(
                df_acquire, stim_treat, left_on="fov", right_on="fov", how="left"
            )

        df_acquire["stim_exposure"] = np.nan

        for fov in df_acquire["fov"].unique():
            fov_data = df_acquire[df_acquire["fov"] == fov]

            stim_pattern = fov_data.iloc[0]

            if isinstance(stim_pattern["stim_timestep"], tuple) and isinstance(
                stim_pattern["stim_exposure_list"], tuple
            ):
                exposure_map = dict(
                    zip(
                        stim_pattern["stim_timestep"],
                        stim_pattern["stim_exposure_list"],
                    )
                )

                for timestep in fov_data["timestep"]:
                    if timestep in exposure_map:
                        mask = (df_acquire["fov"] == fov) & (
                            df_acquire["timestep"] == timestep
                        )
                        df_acquire.loc[mask, "stim_exposure"] = exposure_map[timestep]

        df_acquire["stim"] = df_acquire.apply(
            lambda row: (
                row["timestep"] in row["stim_timestep"] and row["stim_exposure"] > 0
            ),
            axis=1,
        )

    df_acquire = df_acquire.sort_values(by=["time", "fov"]).reset_index(drop=True)
    df_acquire = df_acquire.dropna(axis=1, how="all")
    if add_stim_exposure_group and regular_spacing_between_stimulations:
        spacing_interval = (
            df_acquire["stim_timestep"][0][1] - df_acquire["stim_timestep"][0][0]
        )
        for start in range(0, df_acquire["timestep"].max(), spacing_interval):
            end = start + spacing_interval
            mask = (df_acquire["timestep"] >= start) & (df_acquire["timestep"] < end)
            window = df_acquire.loc[mask, "stim_exposure"]
            value = window.dropna().iloc[0] if window.dropna().size > 0 else np.nan
            df_acquire.loc[mask, "stim_exposure"] = value

    else:
        df_acquire["stim_exposure"] = df_acquire["stim_exposure"].fillna(0)

    return df_acquire


fovs = generate_fov_objects(filename="fovs.json")
fovs

df_acquire = generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=0,
    condition=condition,
)
# df_acquire = apply_stim_treatments_to_df_acquire(
#     df_acquire,
#     stim_exposures_timesteps_phase_1,
#     condition,
#     n_fovs_per_well=n_fovs_per_well,
#     add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
#     regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
# )
df_acquire

Total Experiment Time: 0.001388888888888889h


,fov_object,fov,fov_x,fov_y,fov_z,fov_name,timestep,time,channels,fname,cell_line,phase,phase_id,phase_timestep
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,5133.7,-30138.9,5514.72,0,0,0.0,"({'name': 'Red', 'exposure': 100, 'group': Non...",000_00_00000,test,PreDrug,0,0
1,<rtm_pymmcore.data_structures.Fov object at 0x...,0,5133.7,-30138.9,5514.72,0,1,5.0,"({'name': 'Red', 'exposure': 100, 'group': Non...",000_00_00001,test,PreDrug,0,1


In [6]:
df_acquire_2 = generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_2,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    channel_optocheck=channel_optocheck,
    optocheck_timepoints=optocheck_timepoints,
    phase_name="PostDrug",
    phase_id=1,
    condition=condition,
)
# df_acquire_2 = apply_stim_treatments_to_df_acquire(
#     df_acquire_2,
#     stim_exposures_timesteps_phase_2,
#     condition,
#     n_fovs_per_well=n_fovs_per_well,
#     add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
#     regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
# )
df_acquire_2

Total Experiment Time: 0.002777777777777778h


,fov_object,fov,fov_x,fov_y,fov_z,fov_name,timestep,time,channels,fname,cell_line,optocheck,optocheck_channels,phase,phase_id,phase_timestep
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,5133.7,-30138.9,5514.72,0,0,0.0,"({'name': 'Red', 'exposure': 100, 'group': Non...",000_01_00000,test,False,"({'name': 'Red', 'exposure': 300, 'group': Non...",PostDrug,1,3
1,<rtm_pymmcore.data_structures.Fov object at 0x...,0,5133.7,-30138.9,5514.72,0,1,5.0,"({'name': 'Red', 'exposure': 100, 'group': Non...",000_01_00001,test,False,"({'name': 'Red', 'exposure': 300, 'group': Non...",PostDrug,1,4
2,<rtm_pymmcore.data_structures.Fov object at 0x...,0,5133.7,-30138.9,5514.72,0,2,10.0,"({'name': 'Red', 'exposure': 100, 'group': Non...",000_01_00002,test,False,"({'name': 'Red', 'exposure': 300, 'group': Non...",PostDrug,1,5


### Run experiment

In [9]:
pd.read_parquet("E:\\Alex\\Test_1\\tracks\\000_01_00002.parquet")

,label,x,y,area,fov,fov_x,fov_y,fov_z,fov_name,timestep,...,cell_line,phase,phase_id,phase_timestep,stim,optocheck,time_acquired,particle,fov_timestep,optocheck_channels
0,1,10.874286,31.565714,175.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,False,False,2025-11-10-20:21:35,0,0,None
1,2,24.914729,129.527132,129.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,False,False,2025-11-10-20:21:35,1,0,None
2,3,32.664384,45.472603,146.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,False,False,2025-11-10-20:21:35,2,0,None
3,4,58.102190,22.416058,137.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,False,False,2025-11-10-20:21:35,3,0,None
4,5,59.943750,141.956250,160.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,False,False,2025-11-10-20:21:35,4,0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3720,738,1094.240260,246.045455,154.0,0,5133.7,-30138.9,5514.72,0,2,...,test,PostDrug,1,5,False,False,2025-11-10-20:51:35,745,4,"[{'device_name': None, 'exposure': 300, 'group..."
3721,739,1094.904762,334.744589,231.0,0,5133.7,-30138.9,5514.72,0,2,...,test,PostDrug,1,5,False,False,2025-11-10-20:51:35,744,4,"[{'device_name': None, 'exposure': 300, 'group..."
3722,740,1095.883534,298.208835,249.0,0,5133.7,-30138.9,5514.72,0,2,...,test,PostDrug,1,5,False,False,2025-11-10-20:51:35,746,4,"[{'device_name': None, 'exposure': 300, 'group..."
3723,741,1102.311224,470.403061,196.0,0,5133.7,-30138.9,5514.72,0,2,...,test,PostDrug,1,5,False,False,2025-11-10-20:51:35,747,4,"[{'device_name': None, 'exposure': 300, 'group..."


In [7]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass



mic.run_experiment(df_acquire)

Segmenting image...
add remaining frame related info to df...
Tracking cells...
First frame - initializing linker
Timestep counter: 0
Segmenting image...
add remaining frame related info to df...
Tracking cells...
Subsequent frame - updating linker
Timestep counter: 1


In [1]:
import pandas as pd
db0 = pd.read_parquet("debug_df00_00000.parquet")
db1 = pd.read_parquet("debug_df00_00001.parquet")
db2 = pd.read_parquet("debug_df01_00000.parquet")
db3 = pd.read_parquet("debug_df01_00001.parquet")
db4 = pd.read_parquet("debug_df01_00002.parquet")

In [2]:
db1c = db1[["x", "y"]].copy()
db2c = db2[["x", "y"]].copy()
db3c = db3[["x", "y"]].copy()
db4c = db4[["x", "y"]].copy()
db0c = db0[["x", "y"]].copy()

In [3]:
import trackpy
testl = trackpy.linking.Linker(search_range=15, memory = 3,   adaptive_stop=3, adaptive_step=1)

In [4]:
testl.init_level(db0c.values, 0)
db0["particle"] = testl.particle_ids

In [5]:
testl.next_level(db1c.values, 1)
db1["particle"] = testl.particle_ids

In [6]:
testl.next_level(db2c.values, 2)
db2["particle"]= testl.particle_ids

In [7]:
testl.next_level(db3c.values, 3)
db3["particle"]= testl.particle_ids
testl.next_level(db4c.values, 4)
db4["particle"]= testl.particle_ids

In [8]:
pd.concat([db0, db1, db2, db3, db4], ignore_index=True)

,label,x,y,area,fov,fov_x,fov_y,fov_z,fov_name,timestep,...,cell_line,phase,phase_id,phase_timestep,last_channel,stim,optocheck,time_acquired,particle,optocheck_channels
0,1,10.874286,31.565714,175.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,True,False,False,2025-11-10-20:21:35,0,NaN
1,2,24.914729,129.527132,129.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,True,False,False,2025-11-10-20:21:35,1,NaN
2,3,32.664384,45.472603,146.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,True,False,False,2025-11-10-20:21:35,2,NaN
3,4,58.102190,22.416058,137.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,True,False,False,2025-11-10-20:21:35,3,NaN
4,5,59.943750,141.956250,160.0,0,5133.7,-30138.9,5514.72,0,0,...,test,PreDrug,0,0,True,False,False,2025-11-10-20:21:35,4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3720,738,1094.240260,246.045455,154.0,0,5133.7,-30138.9,5514.72,0,2,...,test,PostDrug,1,5,True,False,False,2025-11-10-20:51:35,745,"[{'device_name': None, 'exposure': 300, 'group..."
3721,739,1094.904762,334.744589,231.0,0,5133.7,-30138.9,5514.72,0,2,...,test,PostDrug,1,5,True,False,False,2025-11-10-20:51:35,744,"[{'device_name': None, 'exposure': 300, 'group..."
3722,740,1095.883534,298.208835,249.0,0,5133.7,-30138.9,5514.72,0,2,...,test,PostDrug,1,5,True,False,False,2025-11-10-20:51:35,746,"[{'device_name': None, 'exposure': 300, 'group..."
3723,741,1102.311224,470.403061,196.0,0,5133.7,-30138.9,5514.72,0,2,...,test,PostDrug,1,5,True,False,False,2025-11-10-20:51:35,747,"[{'device_name': None, 'exposure': 300, 'group..."


In [ ]:
mic.post_experiment()
time.sleep(10)


fovs_i_list = os.listdir(os.path.join(path, "tracks"))
fovs_i_list.sort()
dfs = []

for fov_i in fovs_i_list:

    track_file = os.path.join(path, "tracks", fov_i)
    df = pd.read_parquet(track_file)
    dfs.append(df)

pd.concat(dfs).to_parquet(os.path.join(path, "exp_data.parquet"))

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [ ]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()